In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [9]:
df=pd.read_csv('cardekho_imputated.csv',index_col=0)
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [10]:
col=['car_name','brand']
df=df.drop(col,axis=1)

In [11]:
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [12]:
df.isnull().sum()

model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [13]:
len(df['model'].unique())

120

In [14]:
cat_features=[feature for feature in df.columns if df[feature].dtype=='O']
print(f'length of cat_features:{len(cat_features)}')
num_features=[feature for feature in df.columns if df[feature].dtype!='O']
print(f'length of num_features:{len(num_features)}')
discrete_features=[feature for feature in num_features if len(df[feature].unique())<=25]
print(f'length of discrete features:{len(discrete_features)}')
continous_features=[feature for feature in num_features if len(df[feature].unique())>25]
print(f'length of  continous features:{len(continous_features)}')

length of cat_features:4
length of num_features:7
length of discrete features:2
length of  continous features:5


In [15]:
#independent and dependent features
X=df.drop('selling_price',axis=1)
y=df['selling_price']

In [16]:
#label encoding for model
from sklearn.preprocessing import LabelEncoder
lb=LabelEncoder()
X['model']=lb.fit_transform(X['model'])

In [17]:
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,54,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,118,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,7,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,38,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [22]:
one_hot_cols=['seller_type','fuel_type','transmission_type']
num_features=X.select_dtypes(exclude='object').columns

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

onehotencoder=OneHotEncoder(drop='first')
standard_scaler=StandardScaler()

preprocessing=ColumnTransformer(
    [
        ('onehotencodeing',onehotencoder,one_hot_cols),
        ('StandardScaler',standard_scaler,num_features)
    ],remainder='passthrough'
)

In [23]:
X=preprocessing.fit_transform(X)

In [24]:
#train test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

Model Evaluation

In [25]:
from sklearn.linear_model import LinearRegression,Lasso,Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

In [26]:
def evaluate_metrics(predicted,true):
    r2=r2_score(predicted,true)
    mae=mean_absolute_error(predicted,true)
    mse=mean_squared_error(predicted,true)
    rmse=np.sqrt(mse)
    return r2,mae,mse,rmse

In [29]:
#model training pipeline
models={
    'LinearRegression':LinearRegression(),
    'Lasso':Lasso(),
    'Ridge':Ridge(),
    'KNeighborsRegressor':KNeighborsRegressor(),
    'RandomForestClassifier':RandomForestClassifier(),
    'DecisionTreeClassifier':DecisionTreeClassifier()
}
for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,y_train) #train model

    #prediction
    y_train_pred=model.predict(X_train)
    y_test_pred=model.predict(X_test)

    #evaluations
    r2_training,mae_training,mse_training,rmse_training=evaluate_metrics(y_train_pred,y_train)
    r2_test,mae_test,mse_test,rmse_test=evaluate_metrics(y_test_pred,y_test)

    print(list(models.keys())[i])
    print('-----------------------Training scores-----------------------------')
    print(f'r2:{r2_training}')
    print(f'mae:{mae_training}')
    print(f'rse:{mse_training}')
    print(f'rmse:{rmse_training}')
    print('---------------------------Testing Scores-----------------------------')
    print(f'r2:{r2_test}')
    print(f'mae:{mae_test}')
    print(f'mse:{mse_test}')
    print(f'rmse:{rmse_test}')
    print('-----------------------------------------------------------------')

LinearRegression
-----------------------Training scores-----------------------------
r2:0.3826664077637665
mae:268437.6549149382
rse:312831831093.03534
rmse:559313.714379538
---------------------------Testing Scores-----------------------------
r2:0.5179338847027917
mae:281329.904168813
mse:257613903997.10922
rmse:507556.79878916923
-----------------------------------------------------------------
Lasso
-----------------------Training scores-----------------------------
r2:0.3826614460329324
mae:268436.5960019974
rse:312831842666.6471
rmse:559313.7247257992
---------------------------Testing Scores-----------------------------
r2:0.5179429948553085
mae:281329.3312944437
mse:257612992800.0465
rmse:507555.90115774097
-----------------------------------------------------------------
Ridge
-----------------------Training scores-----------------------------
r2:0.38250967713500017
mae:268393.25894609524
rse:312832747644.5701
rmse:559314.5337326486
---------------------------Testing Scores---

Hyperparameter tunning

In [34]:
#parameters
knn_param={
    'n_neighbors':[2,3,10,20,40,50]
}
rf_param={
    'max_depth':[5,8,10,None,15],
    'n_estimators':[100,200,300,400],
    'min_samples_split':[2,8,15,20],
    'max_features':[5,7,8,6]
}

In [35]:
#random cv params
randomcv_models=[
    ('knn',KNeighborsRegressor(),knn_param),
    ('rf',RandomForestClassifier(),rf_param)
]

In [37]:
#hyperparameter tunning
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import RandomizedSearchCV
model_name={}

for name,model,param in randomcv_models:
    random=RandomizedSearchCV(estimator=model,param_distributions=param,n_jobs=-1,n_iter=100,cv=3)
    random.fit(X_train,y_train)
    model_name[name]=random.best_estimator_

for model in model_name:
    print('-----------best estimator-----------')
    print(model_name[model])

-----------best estimator-----------
KNeighborsRegressor(n_neighbors=2)
-----------best estimator-----------
RandomForestClassifier(max_features=5, min_samples_split=20)
